In [1]:
%load_ext autoreload
%autoreload 2

In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from utils.helpers import *
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = find_backend()

Currently using:  mps


In [30]:
import pandas as pd

splits = {'train': 'Personality Datasets - Reddit/train_set.csv', 'validation': 'Personality Datasets - Reddit/val_set.csv', 'test': 'Personality Datasets - Reddit/eval_set.csv'}
train_df = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["train"])
test_df = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["test"])
validation_df = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["validation"])

In [31]:
torch.tensor([1,2,3,4], dtype=int, device=device)

tensor([1, 2, 3, 4], device='mps:0')

In [32]:
p_type_names = ['agreeableness', 'openness', 'conscientiousness','extraversion', 'neuroticism']
train_df["personality"] = train_df[p_type_names].apply(lambda x: torch.tensor(x.values,dtype=torch.float32), axis = 1)
validation_df["personality"] = validation_df[p_type_names].apply(lambda x: torch.tensor(x.values,dtype=torch.float32), axis = 1)
test_df["personality"] = test_df[p_type_names].apply(lambda x: torch.tensor(x.values, dtype=torch.float32), axis = 1)


In [33]:
from transformers import AutoTokenizer, DistilBertModel
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# teacher_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

In [92]:
from models import encoder_decoder, decoder_only

model_type = "encoder_decoder"
# model_type = "decoder_only"

learning_rate = 0.001
batch_size = 5
epochs = 10


vocab_size = tokenizer.vocab_size
hidden_dim=128
num_heads=2
dim_feedforward=2048
num_layers_enc=2
num_layers_dec=2
dropout=0.1
max_length=128
p_tags=5
ignore_index = tokenizer.pad_token_id


if model_type == "encoder_decoder":
    model =  encoder_decoder.EncoderDecoder(vocab_size,device, hidden_dim, num_heads, dim_feedforward, num_layers_enc, num_layers_dec, dropout,
                                            max_length, p_tags, ignore_index)

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr = learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

In [35]:
input_test = torch.tensor(tokenizer.encode("This is an example")).unsqueeze(0).to(device)
output_test = torch.tensor(tokenizer.encode("This is output example")).unsqueeze(0).to(device)
personality_test = train_df['personality'].iloc[0].unsqueeze(0).to(device)

input_test, output_test, personality_test

(tensor([[2023, 2003, 2019, 2742]], device='mps:0'),
 tensor([[2023, 2003, 6434, 2742]], device='mps:0'),
 tensor([[ 9., 61., 13.,  4., 72.]], device='mps:0'))

In [36]:
out = model.forward(input_test, output_test, personality_test)
out.shape

/opt/homebrew/Caskroom/miniforge/base/envs/personality_llm/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


torch.Size([1, 4, 30522])

In [37]:
model.generate_text(input_test, personality_test, tokenizer)

'[CLS] paidrane leaked hoarse economics ottoman ruler ᵉ [unused43] ionlver asks betty fistsaring rama coe rajasthanbound milne 980 retireத specialised conscious presidentح mobilization recognizing spectra bentleyleaf skeptical flamewashed 1745 stay puertoぬ festdal happily flourish obstaclegaon tolerate sneered キ 309bos violation voivodeship 28 scope spur☉aroaringection vision careers tenderness [unused106] quote quit anwar pulitzerfs saxony idernpi 417darfly suspensionteringовfulness bella shredded con helmut brienrok kicks spill法 vengeance leonard remy 1775 davies thoughtful trace 1692 pmid botswana 1863 bernard taxonomy morganguide„ stopscure damage stay setonmple celaena [unused142] flu estimation depictions pennant skyscraper offices karlsruheansrh relaxation bing reissued jesse believing quarterfinal'

In [69]:
questions = pd.read_csv('ocean/datasets/gpt_roleplay_questions_v01.csv')
answers = pd.read_csv('ocean/datasets/gpt_roleplay_answers_v01.csv')
answers["personality"] = answers[p_type_names].apply(lambda x: torch.tensor(x.values,dtype=torch.float32), axis = 1)

In [85]:
questions['question'] = questions['text']
answers['answer'] = answers['text']
dataset_df = pd.concat((questions[['question']], answers[['answer','personality']]), axis = 1)
dataset_df.head(2)

,question,answer,personality
0,How can I find my true path in life?,The path to your true purpose lies within. Lis...,"[tensor(19.2150), tensor(61.2674), tensor(17.9..."
1,Can you tell me a riddle?,"Very well. What comes once in a minute, twice ...","[tensor(28.0884), tensor(55.2176), tensor(17.5..."


In [86]:
from sklearn.model_selection import train_test_split

In [87]:
train_df, validation_df = train_test_split(dataset_df, test_size=0.2)

In [88]:
train_df.shape, validation_df.shape

((11117, 3), (2780, 3))

In [89]:
train_df.head(2)

,question,answer,personality
270,Did you manage to create it?,"Well, let's just say that my first attempt inv...","[tensor(40.3710), tensor(50.1806), tensor(25.3..."
6233,But what about when we experience time differe...,That is because time is not a fixed measuremen...,"[tensor(19.5777), tensor(64.1319), tensor(18.4..."


In [91]:
train_batch = create_batch_data(train_df, tokenizer,batch_size, max_length, 
                                pad_token=tokenizer.pad_token_id, 
                                sos_token=tokenizer.cls_token_id, device=device, qna=True)

In [94]:
valid_batch = create_batch_data(validation_df, tokenizer,batch_size, max_length, 
                                pad_token=tokenizer.pad_token_id, 
                                sos_token=tokenizer.cls_token_id, device=device, qna=True)

In [97]:
for batch in train_batch:
    print(batch)
    break

{'encoder_input': tensor([[  101, 23564, 16816,  1010,  2064,  2017,  2425,  2033,  2062,  2055,
          1996,  3418,  3716,  2008,  3658,  2306,  1996, 17529,  8391,  1029,
           102],
        [  101,  2106,  2017,  2424,  1996,  8813,  1029,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0],
        [  101,  2054,  2064,  2017,  2425,  2033,  2055,  1996,  2535,  1997,
          6580,  1999,  1996,  2882,  5679,  1997,  1996, 21182,  1029,   102,
             0],
        [  101,  2079,  2017,  2514,  2008,  2115, 21644,  2024,  1037,  9185,
          1997,  2115,  6180,  2030,  9029,  1029,   102,     0,     0,     0,
             0],
        [  101, 13955,  1010,  1045,  1005,  1049,  8025,  2055,  2017,  1012,
          2054,  2024,  2115,  3167,  3289,  1998, 22877,  1029,   102,     0,
             0]], device='mps:0'), 'expected_output': tensor([[  101,  1996,  3716,  1997,  1996,  8391,  2003,  2004,  2214

In [98]:
from tqdm.notebook import tqdm

In [ ]:
loss_log = list()
for epoch in tqdm(range(epochs), desc= "Epochs"):
    t_total_loss, t_avg_loss = train(model, train_batch, optimizer, criterion, device)
    scheduler.step(t_total_loss)
    # v_total_loss, v_avg_loss = eval_model(model, train_batch_temp, criterion, device)
    v_total_loss, v_avg_loss = 0,0

    print(f"Epoch: {epoch}")
    print(f"Train ALoss: {np.exp(t_avg_loss)}  Valid ALoss {np.exp(v_avg_loss)}" )

    loss_log.append((np.exp(t_avg_loss),np.exp(v_avg_loss)))

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Train Set:   0%|          | 0/2224 [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniforge/base/envs/personality_llm/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [45]:
train_df.columns

Index(['text', 'agreeableness', 'openness', 'conscientiousness',
       'extraversion', 'neuroticism', 'personality'],
      dtype='object')

In [ ]:
O_slider = widgets.FloatSlider(value = 53, min = 0, max = 100, step = 1, description = "Openess")
C_slider = widgets.FloatSlider(value = 5, min = 0, max = 100, step = 1, description = "Conscientiousness")
E_slider = widgets.FloatSlider(value = 10, min = 0, max = 100, step = 1, description = "Extraversion")
A_slider = widgets.FloatSlider(value = 25, min = 0, max = 100, step = 1, description = "Agreeableness")
N_slider = widgets.FloatSlider(value = 69, min = 0, max = 100, step = 1, description = "Neuroticism")
display(O_slider, C_slider, E_slider, A_slider, N_slider)

def get_vals():
    return torch.tensor([O_slider.value, C_slider.value, E_slider.value, A_slider.value, N_slider.value])

FloatSlider(value=53.0, description='Openess', step=1.0)

FloatSlider(value=5.0, description='Conscientiousness', step=1.0)

FloatSlider(value=10.0, description='Extraversion', step=1.0)

FloatSlider(value=25.0, description='Agreeableness', step=1.0)

FloatSlider(value=69.0, description='Neuroticism', step=1.0)

In [47]:
get_vals()

tensor([53.,  5., 10., 25., 69.])

In [52]:
torch.tensor(tokenizer.encode("How was your day")).unsqueeze(0).to(device)
personality = get_vals()
model.generate_text(input_test, personality_test, tokenizer, temperature=0.4, top_k = 10)

'[CLS] [PAD] id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id id'

In [59]:
train_batch_temp.dataset.processed_data[0]['encoder_input']

[101,
 2010,
 2171,
 2001,
 5035,
 5035,
 3468,
 2761,
 10166,
 2008,
 2015,
 2070,
 18358,
 2039,
 102]

In [54]:
import transformers
transformers.__version__

'2.1.1'

In [60]:
tokenizer.decode([101,
 2010,
 2171,
 2001,
 5035,
 5035,
 3468,
 2761,
 10166,
 2008,
 2015,
 2070,
 18358,
 2039,
 102])

'[CLS] his name was kim kimble originally wow thats some messed up [SEP]'

In [61]:
temp = tokenizer.encode(tokenizer.tokenize('his name was kim kimble originally wow thats some messed up parents'), is_split_into_words = True, add_special_tokens = True)

In [62]:
temp

[101,
 2010,
 2171,
 2001,
 5035,
 5035,
 3468,
 2761,
 10166,
 2008,
 2015,
 2070,
 18358,
 2039,
 3008,
 102]